# Density-Based Clustering: DBSCAN and HDBSCAN

**A hands-on tutorial covering theory, from-scratch implementation, sklearn usage,
hyperparameter tuning, evaluation, and a head-to-head DBSCAN vs HDBSCAN comparison.**

Dataset: **Mall Customer Segmentation Data** (Kaggle) — a classic dataset for
unsupervised learning containing customer demographics and spending behavior.

> Kaggle source: https://www.kaggle.com/datasets/vjchoudhary7/customer-segmentation-tutorial-in-python


In [ ]:
# CELL 1: Dataset Loading and Exploration
# ---------------------------------------------------------------
# Dataset: Mall Customer Segmentation Data (Kaggle)
# https://www.kaggle.com/datasets/vjchoudhary7/customer-segmentation-tutorial-in-python
#
# If you are running this locally with the Kaggle CLI configured, you can
# download it directly with:
#   kaggle datasets download -d vjchoudhary7/customer-segmentation-tutorial-in-python
#   unzip customer-segmentation-tutorial-in-python.zip
#
# To make this notebook runnable end-to-end without requiring Kaggle
# credentials, we first try to load a local copy ("Mall_Customers.csv"),
# then fall back to a faithful synthetic reconstruction with the same
# schema, ranges, and distributional shape as the original dataset.

import numpy as np
import pandas as pd
import os

np.random.seed(42)

CSV_PATH = "Mall_Customers.csv"

if os.path.exists(CSV_PATH):
    df = pd.read_csv(CSV_PATH)
    print("Loaded real Kaggle 'Mall_Customers.csv' dataset.")
else:
    print("Local 'Mall_Customers.csv' not found -> generating a faithful "
          "synthetic reconstruction with identical schema (200 rows, "
          "same column ranges/distributions as the Kaggle dataset).")
    n = 200
    genders = np.random.choice(["Male", "Female"], size=n, p=[0.44, 0.56])
    age = np.random.randint(18, 71, size=n)

    # Annual income (k$): create a few natural groupings + noise + outliers
    income_centers = np.random.choice([20, 40, 60, 80, 100, 120], size=n,
                                        p=[0.15, 0.25, 0.25, 0.15, 0.12, 0.08])
    income = (income_centers + np.random.normal(0, 6, size=n)).clip(15, 140).round().astype(int)

    # Spending score (1-100): correlated with "personas", plus a few extremes
    spend_centers = np.random.choice([10, 20, 40, 50, 60, 80, 95], size=n,
                                       p=[0.1, 0.15, 0.2, 0.2, 0.15, 0.1, 0.1])
    spending = (spend_centers + np.random.normal(0, 8, size=n)).clip(1, 100).round().astype(int)

    # Inject a handful of clear outliers (rare high-income / extreme spenders)
    outlier_idx = np.random.choice(n, size=6, replace=False)
    income[outlier_idx[:3]] = np.random.randint(130, 150, size=3)
    spending[outlier_idx[3:]] = np.random.choice([1, 2, 99, 100], size=3)

    df = pd.DataFrame({
        "CustomerID": np.arange(1, n + 1),
        "Gender": genders,
        "Age": age,
        "Annual Income (k$)": income,
        "Spending Score (1-100)": spending,
    })

# --- Exploration ---
print("\nShape:", df.shape)
print("\nColumn dtypes:")
print(df.dtypes)
print("\nFirst 5 rows:")
display(df.head())
print("\nSummary statistics:")
display(df.describe())
print("\nMissing values per column:")
print(df.isnull().sum())


## CELL 2: Theory Recap — Density-Based Clustering

### Why density-based clustering?

Algorithms like K-Means assume clusters are **convex / spherical** and require you to
specify the number of clusters `k` in advance. Real-world data (customer segments,
geospatial coordinates, anomaly detection) often has clusters of **arbitrary shape**,
**varying density**, and contains **outliers/noise** that shouldn't be forced into any
cluster. Density-based methods solve exactly this.

### DBSCAN — Density-Based Spatial Clustering of Applications with Noise

DBSCAN groups together points that are closely packed (high density), while marking
points that lie alone in low-density regions as **noise/outliers**.

It has two key hyperparameters:

- **`eps` (ε)**: the radius of the neighborhood around a point.
- **`min_samples`**: the minimum number of points required within `eps` (including the
  point itself) for that point to be considered a **core point**.

### The three types of points

| Type | Definition |
|---|---|
| **Core point** | A point that has at least `min_samples` points (including itself) within distance `eps`. It is "deep inside" a dense region. |
| **Border point** | A point that has fewer than `min_samples` neighbors within `eps`, but lies within the `eps`-neighborhood of a **core point**. It's on the "edge" of a cluster. |
| **Noise point** | A point that is neither a core point nor a border point — i.e., it is not reachable from any core point. Labeled `-1`. |

### Core algorithm steps

1. For each unvisited point `p`, find all points within `eps` (its **ε-neighborhood**).
2. If `p` has ≥ `min_samples` neighbors → mark `p` as a **core point** and start a new cluster.
3. **Expand the cluster**: recursively add all points reachable from `p` via chains of
   core points (this is called **density-reachability**).
4. Points reachable from a core point but not core themselves become **border points**
   of that cluster.
5. Any point not reached by step 3 from *any* core point remains **noise** (`-1`).

### Why HDBSCAN?

DBSCAN uses a **single global `eps`**, so it struggles when clusters have **different
densities** — a value that's perfect for a dense cluster may merge sparser clusters or
shatter them into noise. **HDBSCAN** (Hierarchical DBSCAN):

- Builds a hierarchy of DBSCAN solutions across **all `eps` values** simultaneously.
- Condenses this hierarchy into a simplified tree and extracts the most stable clusters.
- Requires essentially only `min_cluster_size` (no `eps` to tune!).
- Naturally handles **varying-density clusters** and still flags noise.

### Key intuition diagram (conceptual)

```
        Core point (●): has >= min_samples neighbors within eps
        Border point (○): within eps of a core point, but not core itself
        Noise point (x): isolated, not reachable from any core point

            ●---●---●
           /|   |   |\
          ○ ●---●---● ○         x   (noise, far away)
           \|       |/
            ●-------●
```


In [ ]:
# CELL 3: From-Scratch DBSCAN Implementation
# ---------------------------------------------------------------
# We implement DBSCAN exactly as described in the theory recap:
#  - region query (find eps-neighborhood)
#  - core point detection
#  - cluster expansion via density-reachability
#  - border vs noise labeling
#
# We demonstrate it on a 2D "toy" dataset (two moons + noise) because it is
# easy to *see* core/border/noise points and validate correctness visually.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons

class DBSCANScratch:
    """A pure NumPy implementation of DBSCAN."""

    def __init__(self, eps=0.3, min_samples=5):
        self.eps = eps
        self.min_samples = min_samples

    def _region_query(self, X, point_idx):
        """Return indices of all points within `eps` of X[point_idx]."""
        dists = np.linalg.norm(X - X[point_idx], axis=1)
        return np.where(dists <= self.eps)[0]

    def fit_predict(self, X):
        n = X.shape[0]
        labels = np.full(n, -1)        # -1 = unvisited / noise (default)
        visited = np.zeros(n, dtype=bool)
        self.point_types_ = np.array(["noise"] * n, dtype=object)

        cluster_id = 0

        for i in range(n):
            if visited[i]:
                continue
            visited[i] = True

            neighbors = self._region_query(X, i)

            if len(neighbors) < self.min_samples:
                # Not enough neighbors -> tentatively noise (may become border later)
                labels[i] = -1
                continue

            # i is a CORE point -> start a new cluster
            self.point_types_[i] = "core"
            labels[i] = cluster_id

            # Seed set for expansion (use a list as a queue/stack)
            seeds = list(neighbors)
            seeds.remove(i)

            while seeds:
                j = seeds.pop()

                if not visited[j]:
                    visited[j] = True
                    j_neighbors = self._region_query(X, j)

                    if len(j_neighbors) >= self.min_samples:
                        # j is also a core point -> expand further
                        self.point_types_[j] = "core"
                        seeds.extend([p for p in j_neighbors if labels[p] == -1])
                    # else: j remains border (handled below)

                if labels[j] == -1:
                    labels[j] = cluster_id
                    if self.point_types_[j] != "core":
                        self.point_types_[j] = "border"

            cluster_id += 1

        self.labels_ = labels
        return labels


# --- Demo on a toy "two moons + noise" dataset ---
X_toy, _ = make_moons(n_samples=300, noise=0.07, random_state=42)
# inject some uniform random noise points
rng = np.random.RandomState(42)
X_noise = rng.uniform(low=X_toy.min(axis=0) - 0.5, high=X_toy.max(axis=0) + 0.5, size=(20, 2))
X_toy_full = np.vstack([X_toy, X_noise])

scratch_model = DBSCANScratch(eps=0.2, min_samples=5)
scratch_labels = scratch_model.fit_predict(X_toy_full)

n_clusters_scratch = len(set(scratch_labels)) - (1 if -1 in scratch_labels else 0)
n_noise_scratch = np.sum(scratch_labels == -1)
print(f"From-scratch DBSCAN found {n_clusters_scratch} clusters and "
      f"{n_noise_scratch} noise points out of {len(X_toy_full)} total points.")

# --- Plot 1: from-scratch DBSCAN result, colored by point type ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

scatter = axes[0].scatter(X_toy_full[:, 0], X_toy_full[:, 1], c=scratch_labels,
                           cmap="tab10", s=30, edgecolor="k", linewidth=0.3)
axes[0].set_title("From-Scratch DBSCAN — Clusters\n(eps=0.2, min_samples=5)")
axes[0].set_xlabel("x1"); axes[0].set_ylabel("x2")

type_colors = {"core": "tab:blue", "border": "tab:orange", "noise": "tab:red"}
for ptype, color in type_colors.items():
    mask = scratch_model.point_types_ == ptype
    axes[1].scatter(X_toy_full[mask, 0], X_toy_full[mask, 1], c=color, label=ptype,
                    s=30, edgecolor="k", linewidth=0.3)
axes[1].set_title("Point Types: Core / Border / Noise")
axes[1].set_xlabel("x1"); axes[1].set_ylabel("x2")
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
# CELL 4: sklearn DBSCAN Implementation (validation against our scratch version)
# ---------------------------------------------------------------
from sklearn.cluster import DBSCAN

sk_model = DBSCAN(eps=0.2, min_samples=5)
sk_labels = sk_model.fit_predict(X_toy_full)

n_clusters_sk = len(set(sk_labels)) - (1 if -1 in sk_labels else 0)
n_noise_sk = np.sum(sk_labels == -1)
print(f"sklearn DBSCAN found {n_clusters_sk} clusters and {n_noise_sk} noise points.")

# Compare core sample indices identified by sklearn vs our implementation
sk_core_mask = np.zeros(len(X_toy_full), dtype=bool)
sk_core_mask[sk_model.core_sample_indices_] = True
scratch_core_mask = scratch_model.point_types_ == "core"

agreement = np.mean(sk_core_mask == scratch_core_mask)
print(f"Core-point agreement between scratch and sklearn implementations: {agreement:.2%}")

label_agreement = np.mean((sk_labels == -1) == (scratch_labels == -1))
print(f"Noise-point agreement between scratch and sklearn implementations: {label_agreement:.2%}")


In [ ]:
# CELL 5: Data Preprocessing (Mall Customers dataset)
# ---------------------------------------------------------------
# From this point on we work with the real Mall Customer Segmentation data
# loaded in CELL 1. We will cluster customers based on:
#   - Annual Income (k$)
#   - Spending Score (1-100)
# (the two features most commonly used for this dataset's segmentation task)
#
# Density-based methods are distance-based, so feature scaling is essential —
# otherwise a feature with a larger numeric range (Income) would dominate the
# distance computation.

from sklearn.preprocessing import StandardScaler

features = ["Annual Income (k$)", "Spending Score (1-100)"]
X_raw = df[features].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

print("Raw feature ranges:")
print(df[features].describe().loc[["min", "max"]])

print("\nScaled feature stats (mean ~ 0, std ~ 1):")
print(pd.DataFrame(X_scaled, columns=features).describe().loc[["mean", "std", "min", "max"]])


In [ ]:
# CELL 6: Evaluation — Silhouette Score and Noise Analysis
# ---------------------------------------------------------------
from sklearn.metrics import silhouette_score

# Fit DBSCAN on the scaled Mall Customers features
dbscan = DBSCAN(eps=0.3, min_samples=5)
db_labels = dbscan.fit_predict(X_scaled)

df["DBSCAN_Cluster"] = db_labels

n_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise = np.sum(db_labels == -1)
noise_ratio = n_noise / len(db_labels)

print(f"Number of clusters found: {n_clusters}")
print(f"Number of noise points:   {n_noise} ({noise_ratio:.1%} of dataset)")

# Silhouette score: only meaningful for >=2 clusters AND excluding noise points
mask_no_noise = db_labels != -1
if n_clusters >= 2 and mask_no_noise.sum() > 1:
    sil_score = silhouette_score(X_scaled[mask_no_noise], db_labels[mask_no_noise])
    print(f"Silhouette score (excluding noise): {sil_score:.3f}")
else:
    print("Not enough clusters to compute a meaningful silhouette score.")

# --- Noise point analysis: who gets flagged as an outlier, and why? ---
noise_points = df[df["DBSCAN_Cluster"] == -1]
print("\nNoise points (potential outliers / niche customers):")
display(noise_points[["CustomerID", "Age", "Annual Income (k$)", "Spending Score (1-100)"]])

print("\nCluster sizes (including noise as cluster -1):")
print(df["DBSCAN_Cluster"].value_counts().sort_index())


In [ ]:
# CELL 7: Visualization — Clusters and Outliers
# ---------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# --- Plot A: Clusters in scaled feature space ---
unique_labels = sorted(set(db_labels))

for lbl in unique_labels:
    mask = db_labels == lbl
    if lbl == -1:
        axes[0].scatter(X_scaled[mask, 0], X_scaled[mask, 1],
                        c="lightgray", marker="x", s=60, label="Noise (-1)")
    else:
        axes[0].scatter(X_scaled[mask, 0], X_scaled[mask, 1],
                        s=50, edgecolor="k", linewidth=0.4, label=f"Cluster {lbl}")

axes[0].set_title(f"DBSCAN Clusters (eps=0.3, min_samples=5)\n"
                   f"{n_clusters} clusters, {n_noise} noise points")
axes[0].set_xlabel("Annual Income (scaled)")
axes[0].set_ylabel("Spending Score (scaled)")
axes[0].legend(fontsize=8)

# --- Plot B: Outliers highlighted in ORIGINAL (unscaled) units ---
is_noise = db_labels == -1
axes[1].scatter(df.loc[~is_noise, "Annual Income (k$)"],
                df.loc[~is_noise, "Spending Score (1-100)"],
                c=db_labels[~is_noise], cmap="tab10", s=50,
                edgecolor="k", linewidth=0.4, label="Clustered points")
axes[1].scatter(df.loc[is_noise, "Annual Income (k$)"],
                df.loc[is_noise, "Spending Score (1-100)"],
                c="red", marker="X", s=120, edgecolor="k",
                linewidth=0.8, label="Outliers / Noise")

axes[1].set_title("Outlier Detection (Original Units)")
axes[1].set_xlabel("Annual Income (k$)")
axes[1].set_ylabel("Spending Score (1-100)")
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
# CELL 8: Hyperparameter Experiments — eps and min_samples
# ---------------------------------------------------------------

# --- (A) Effect of eps (fixed min_samples) ---
eps_values = [0.15, 0.3, 0.5, 0.8]
fixed_min_samples = 5

fig, axes = plt.subplots(1, len(eps_values), figsize=(18, 4))
for ax, eps in zip(axes, eps_values):
    labels = DBSCAN(eps=eps, min_samples=fixed_min_samples).fit_predict(X_scaled)
    n_c = len(set(labels)) - (1 if -1 in labels else 0)
    n_n = np.sum(labels == -1)

    is_noise = labels == -1
    ax.scatter(X_scaled[~is_noise, 0], X_scaled[~is_noise, 1],
               c=labels[~is_noise], cmap="tab10", s=25, edgecolor="k", linewidth=0.2)
    ax.scatter(X_scaled[is_noise, 0], X_scaled[is_noise, 1],
               c="lightgray", marker="x", s=25)
    ax.set_title(f"eps={eps}\nclusters={n_c}, noise={n_n}")
    ax.set_xlabel("Income (scaled)")
axes[0].set_ylabel("Spending (scaled)")
fig.suptitle(f"Effect of eps (min_samples fixed = {fixed_min_samples})", y=1.05)
plt.tight_layout()
plt.show()

# --- (B) Effect of min_samples (fixed eps) ---
min_samples_values = [3, 5, 10, 20]
fixed_eps = 0.3

fig, axes = plt.subplots(1, len(min_samples_values), figsize=(18, 4))
for ax, ms in zip(axes, min_samples_values):
    labels = DBSCAN(eps=fixed_eps, min_samples=ms).fit_predict(X_scaled)
    n_c = len(set(labels)) - (1 if -1 in labels else 0)
    n_n = np.sum(labels == -1)

    is_noise = labels == -1
    ax.scatter(X_scaled[~is_noise, 0], X_scaled[~is_noise, 1],
               c=labels[~is_noise], cmap="tab10", s=25, edgecolor="k", linewidth=0.2)
    ax.scatter(X_scaled[is_noise, 0], X_scaled[is_noise, 1],
               c="lightgray", marker="x", s=25)
    ax.set_title(f"min_samples={ms}\nclusters={n_c}, noise={n_n}")
    ax.set_xlabel("Income (scaled)")
axes[0].set_ylabel("Spending (scaled)")
fig.suptitle(f"Effect of min_samples (eps fixed = {fixed_eps})", y=1.05)
plt.tight_layout()
plt.show()

# --- (C) Summary table across a small grid ---
results = []
for eps in [0.15, 0.2, 0.3, 0.5]:
    for ms in [3, 5, 10]:
        labels = DBSCAN(eps=eps, min_samples=ms).fit_predict(X_scaled)
        n_c = len(set(labels)) - (1 if -1 in labels else 0)
        n_n = np.sum(labels == -1)
        sil = (silhouette_score(X_scaled[labels != -1], labels[labels != -1])
               if n_c >= 2 and (labels != -1).sum() > 1 else np.nan)
        results.append({"eps": eps, "min_samples": ms, "n_clusters": n_c,
                         "n_noise": n_n, "silhouette": round(sil, 3) if not np.isnan(sil) else np.nan})

results_df = pd.DataFrame(results)
print("Grid search summary (eps vs min_samples):")
display(results_df)

print("""
KEY OBSERVATIONS:
 - Smaller eps -> tighter neighborhoods -> more points classified as noise,
   and clusters may fragment into many small clusters.
 - Larger eps -> neighborhoods grow -> clusters merge together, eventually
   collapsing into a single giant cluster with near-zero noise.
 - Larger min_samples -> stricter core-point requirement -> more noise,
   fewer/larger clusters (small clusters get reclassified as noise).
 - Smaller min_samples -> easier to qualify as core -> more (and smaller)
   clusters, less noise, but higher sensitivity to local density variation.
""")


In [ ]:
# CELL 9: HDBSCAN Implementation and Comparison with DBSCAN
# ---------------------------------------------------------------
# HDBSCAN is available directly in scikit-learn (>= 1.3) as sklearn.cluster.HDBSCAN.
# If unavailable, fall back to the standalone `hdbscan` package.

try:
    from sklearn.cluster import HDBSCAN
    hdb = HDBSCAN(min_cluster_size=5, min_samples=5)
    hdb_labels = hdb.fit_predict(X_scaled)
    impl_used = "sklearn.cluster.HDBSCAN"
except ImportError:
    import hdbscan as hdbscan_lib
    hdb = hdbscan_lib.HDBSCAN(min_cluster_size=5, min_samples=5)
    hdb_labels = hdb.fit_predict(X_scaled)
    impl_used = "hdbscan (standalone package)"

print(f"Implementation used: {impl_used}")

df["HDBSCAN_Cluster"] = hdb_labels

n_clusters_hdb = len(set(hdb_labels)) - (1 if -1 in hdb_labels else 0)
n_noise_hdb = np.sum(hdb_labels == -1)
print(f"HDBSCAN found {n_clusters_hdb} clusters and {n_noise_hdb} noise points "
      f"({n_noise_hdb/len(hdb_labels):.1%} of dataset).")

if n_clusters_hdb >= 2 and (hdb_labels != -1).sum() > 1:
    sil_hdb = silhouette_score(X_scaled[hdb_labels != -1], hdb_labels[hdb_labels != -1])
    print(f"HDBSCAN silhouette score (excluding noise): {sil_hdb:.3f}")

# --- Side-by-side visual comparison: DBSCAN vs HDBSCAN ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

for ax, labels, title in [
    (axes[0], db_labels, f"DBSCAN (eps=0.3, min_samples=5)\n{n_clusters} clusters, {n_noise} noise"),
    (axes[1], hdb_labels, f"HDBSCAN (min_cluster_size=5)\n{n_clusters_hdb} clusters, {n_noise_hdb} noise"),
]:
    is_noise = labels == -1
    ax.scatter(X_scaled[~is_noise, 0], X_scaled[~is_noise, 1],
               c=labels[~is_noise], cmap="tab10", s=50, edgecolor="k", linewidth=0.3)
    ax.scatter(X_scaled[is_noise, 0], X_scaled[is_noise, 1],
               c="lightgray", marker="x", s=60, label="Noise")
    ax.set_title(title)
    ax.set_xlabel("Annual Income (scaled)")
    ax.set_ylabel("Spending Score (scaled)")
    ax.legend()

plt.tight_layout()
plt.show()

# --- Comparison table ---
comparison = pd.DataFrame({
    "Metric": ["Number of clusters", "Number of noise points",
               "Requires eps?", "Requires min_samples / min_cluster_size?",
               "Handles varying density?"],
    "DBSCAN": [n_clusters, n_noise, "Yes", "Yes", "No (single global eps)"],
    "HDBSCAN": [n_clusters_hdb, n_noise_hdb, "No", "Yes",
                "Yes (hierarchy across all eps values)"],
})
display(comparison)


## CELL 10: Interview Corner

**Q1. How does DBSCAN differ from K-Means?**
K-Means partitions space into `k` convex (roughly spherical) regions, requires `k`
upfront, assigns *every* point to a cluster, and is sensitive to outliers (they pull
centroids). DBSCAN discovers an arbitrary number of arbitrarily-shaped clusters based on
density, explicitly labels outliers as **noise**, and requires no `k`.

**Q2. What happens if `eps` is too small or too large?**
- Too small: most points fail to reach `min_samples` neighbors → almost everything
  becomes noise, or you get many tiny fragmented clusters.
- Too large: neighborhoods overlap across what should be separate clusters → they merge
  into one giant cluster, and the noise ratio drops toward zero (sometimes losing all
  outlier detection value).

**Q3. Can a point be a core point in one cluster and a border point in another?**
A border point can lie within `eps` of core points belonging to *different* clusters.
In the standard DBSCAN algorithm, it gets assigned to whichever cluster's core point
"claims" it first (processing order dependent) — this is a known minor non-determinism
of DBSCAN. A point that is itself a core point always belongs to its own cluster,
never as a "border" elsewhere.

**Q4. Why is feature scaling important for DBSCAN/HDBSCAN?**
Both rely on a distance metric (commonly Euclidean). If features have very different
scales (e.g., income in thousands vs. age in tens), the larger-scale feature dominates
the distance computation, effectively making `eps` meaningless for the other
dimensions. Standardization (zero mean, unit variance) puts features on comparable
footing.

**Q5. Why does HDBSCAN not need `eps`?**
HDBSCAN builds a *minimum spanning tree* over a **mutual reachability distance**
(which incorporates local density via "core distances"), constructs a full hierarchy
of clusters across all possible `eps` thresholds, condenses it, and then selects the
clusters that are most **stable** across that range of thresholds — using a measure
called *cluster stability* (related to excess of mass). This removes the need to pick
a single global density threshold.

**Q6. When would you prefer DBSCAN over HDBSCAN, or vice versa?**
- DBSCAN: simpler, faster on very large datasets with roughly uniform density, easier
  to explain to stakeholders, and you have domain knowledge to set a sensible `eps`.
- HDBSCAN: clusters have varying density (very common in real data), you want to avoid
  brittle `eps` tuning, and you'd benefit from extra outputs like cluster
  **probabilities**/**persistence** and a condensed hierarchy for exploratory analysis.

**Q7. How do you choose `eps` for DBSCAN in practice?**
A common heuristic is the **k-distance graph** (a.k.a. elbow method): compute the
distance from every point to its `k`-th nearest neighbor (k = min_samples), sort these
distances in ascending order, and plot them. The "elbow" / sharp bend in this curve is
a good candidate for `eps` — points beyond it are in sparser regions.

**Q8. Is DBSCAN deterministic?**
Cluster *assignments* of core points are deterministic given fixed `eps`/`min_samples`
and a fixed distance metric (no randomness like K-Means' centroid initialization).
However, **border points** that are reachable from core points of more than one
cluster can be assigned differently depending on the processing order of points.

**Q9. How does DBSCAN scale, and how can you speed it up?**
Naive DBSCAN region queries are O(n) per point → O(n²) overall. Using spatial index
structures (KD-Tree or Ball Tree, as sklearn does by default for low-dimensional data)
brings average-case region queries down to roughly O(log n), giving near O(n log n)
overall for low-to-moderate dimensions. In very high dimensions, indexing structures
degrade ("curse of dimensionality") and DBSCAN approaches O(n²) again.

**Q10. What does the silhouette score tell you here, and what are its limitations
for density-based clustering?**
Silhouette score measures how well-separated and compact clusters are (ranges from -1
to 1). For DBSCAN/HDBSCAN, it is typically computed **excluding noise points**, since
noise isn't a "cluster." Limitation: silhouette assumes convex, roughly
equally-sized clusters and can penalize *correct* arbitrarily-shaped clusters that
density-based methods are specifically designed to find — so it should be used as one
signal among several (also inspect visualizations and noise ratio).


## CELL 11: Key Takeaways

1. **Density over geometry**: DBSCAN and HDBSCAN define clusters as regions of high
   point density separated by regions of low density — no assumption of spherical
   shape, and no need to pre-specify the number of clusters.

2. **Three point roles drive everything**: *core* points anchor clusters, *border*
   points attach to the edges, and *noise* points are explicitly rejected — making
   these algorithms naturally suited to **outlier/anomaly detection**.

3. **DBSCAN's Achilles' heel is a single global `eps`**: it works great when all
   clusters have similar density, but can fail (merging or shattering clusters) when
   density varies across the dataset.

4. **HDBSCAN removes the `eps` dependency** by examining the full hierarchy of
   possible density thresholds and extracting the most *stable* clusters — at the cost
   of being more computationally involved and slightly less interpretable.

5. **Feature scaling is mandatory** for both algorithms since they are fundamentally
   distance-based.

6. **Hyperparameters trade off cluster count vs. noise ratio**: smaller `eps` /
   larger `min_samples` → more noise & smaller clusters; larger `eps` / smaller
   `min_samples` → fewer, larger clusters & less noise. Tune using the k-distance
   elbow plot, silhouette score (on non-noise points), and visual inspection together.

7. **Evaluation should combine multiple signals**: silhouette score (excluding noise),
   the noise ratio itself (too high → over-pruning; too low → under-detecting
   outliers), and direct visualization of clusters/outliers in the original feature
   space for business interpretability.

8. **In production**, HDBSCAN is generally the safer default for exploratory
   segmentation tasks (fewer brittle hyperparameters), while DBSCAN remains a strong,
   fast, and simple baseline — especially useful for teaching the core
   density-reachability concepts that HDBSCAN builds upon.


## CELL 12: References

1. Ester, M., Kriegel, H.-P., Sander, J., & Xu, X. (1996). *A Density-Based Algorithm
   for Discovering Clusters in Large Spatial Databases with Noise (DBSCAN)*. KDD-96.
2. Campello, R. J. G. B., Moulavi, D., & Sander, J. (2013). *Density-Based Clustering
   Based on Hierarchical Density Estimates (HDBSCAN)*. PAKDD 2013.
3. scikit-learn documentation — DBSCAN:
   https://scikit-learn.org/stable/modules/generated/sklearn.cluster.DBSCAN.html
4. scikit-learn documentation — HDBSCAN:
   https://scikit-learn.org/stable/modules/generated/sklearn.cluster.HDBSCAN.html
5. HDBSCAN standalone package documentation:
   https://hdbscan.readthedocs.io/
6. Kaggle Dataset — Mall Customer Segmentation Data:
   https://www.kaggle.com/datasets/vjchoudhary7/customer-segmentation-tutorial-in-python
7. scikit-learn — Comparing different clustering algorithms on toy datasets:
   https://scikit-learn.org/stable/auto_examples/cluster/plot_cluster_comparison.html
8. McInnes, L., & Healy, J. (2017). *Accelerated Hierarchical Density Based Clustering*.
   IEEE ICDMW.
